In [ ]:

# # Task 1: Data Exploration and Enrichment
# Ethiopia Financial Inclusion Forecasting — Selam Analytics



In [1]:

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data_loader import load_unified_data, load_reference_codes, validate_schema, summarize # type: ignore

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)



In [2]:
df = load_unified_data()
ref_codes = load_reference_codes()

print(f"Unified dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Reference codes: {ref_codes.shape[0]} rows")
df.head()


Unified dataset: 57 rows, 35 columns
Reference codes: 71 rows


/home/sumeya/Documents/ai project/10x academi/ethiopia-fi-forecast/notebooks/../src/data_loader.py:154: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors="coerce")


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes,parent_id
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaT,NaT,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,Baseline year,NaN,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaT,NaT,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,Gender disaggregated,NaN,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,Gender disaggregated,NaN,NaN


In [4]:
issues = validate_schema(df, ref_codes)
if issues:
    for k, v in issues.items():
        print(f"{k}:")
        print(v)
else:
    print("No schema issues detected.")

No schema issues detected.


In [6]:
summary = summarize(df)
for name, counts in summary.items():
    print(f"--- {name} ---")
    print(counts)
    print()

--- by_record_type ---
record_type
observation    30
impact_link    14
event          10
target          3
Name: count, dtype: int64

--- by_pillar ---
pillar
ACCESS           20
USAGE            17
NaN              10
GENDER            6
AFFORDABILITY     4
Name: count, dtype: int64

--- by_source_type ---
source_type
operator      15
NaN           14
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64

--- by_confidence ---
confidence
high      44
medium    13
Name: count, dtype: int64

--- by_indicator_code ---
indicator_code
NaN                   14
ACC_OWNERSHIP          7
ACC_FAYDA              4
ACC_MM_ACCOUNT         2
ACC_4G_COV             2
USG_P2P_COUNT          2
GEN_GAP_ACC            2
GEN_MM_SHARE           2
ACC_MOBILE_PEN         1
USG_P2P_VALUE          1
USG_ATM_COUNT          1
USG_ATM_VALUE          1
USG_CROSSOVER          1
USG_TELEBIRR_USERS     1
USG_TELEBIRR_VALUE     1
USG_MPESA_USER

In [7]:
obs = df[df['record_type'] == 'observation'].copy()
print("Observation date range:", obs['observation_date'].min(), "to", obs['observation_date'].max())
obs['year'] = obs['observation_date'].dt.year
obs['year'].value_counts().sort_index()

# === Cell 7: Indicator coverage ===
indicator_coverage = (
    obs.groupby(['indicator_code', 'indicator'])
       .agg(n_observations=('value_numeric', 'count'),
            first_year=('year', 'min'),
            last_year=('year', 'max'))
       .reset_index()
       .sort_values('n_observations', ascending=False)
)
indicator_coverage


Observation date range: 2014-12-31 00:00:00 to 2025-12-31 00:00:00


,indicator_code,indicator,n_observations,first_year,last_year
4,ACC_OWNERSHIP,Account Ownership Rate,6,2014,2024
1,ACC_FAYDA,Fayda Digital ID Enrollment,3,2024,2025
0,ACC_4G_COV,4G Population Coverage,2,2023,2025
2,ACC_MM_ACCOUNT,Mobile Money Account Rate,2,2021,2024
6,GEN_GAP_ACC,Account Ownership Gender Gap,2,2021,2024
15,USG_P2P_COUNT,P2P Transaction Count,2,2024,2025
3,ACC_MOBILE_PEN,Mobile Subscription Penetration,1,2025,2025
7,GEN_GAP_MOBILE,Mobile Phone Gender Gap,1,2024,2024
8,GEN_MM_SHARE,Female Mobile Money Account Share,1,2024,2024
9,USG_ACTIVE_RATE,Mobile Money Activity Rate,1,2024,2024


In [8]:
events = df[df['record_type'] == 'event'].copy()
events[['record_id', 'observation_date', 'category', 'source_name', 'original_text']].sort_values('observation_date')


,record_id,observation_date,category,source_name,original_text
33,EVT_0001,2021-05-17,product_launch,Ethio Telecom,First major mobile money service in Ethiopia
41,EVT_0009,2021-09-01,policy,NBE,5-year national financial inclusion strategy
34,EVT_0002,2022-08-01,market_entry,News,End of state telecom monopoly
35,EVT_0003,2023-08-01,product_launch,Safaricom,Second mobile money entrant
36,EVT_0004,2024-01-01,infrastructure,NIDP,National biometric digital ID system
37,EVT_0005,2024-07-29,policy,NBE,Birr float introduced
38,EVT_0006,2024-10-01,milestone,EthSwitch,Historic: digital > cash for first time
39,EVT_0007,2025-10-27,partnership,EthSwitch,Full interoperability for M-Pesa
42,EVT_0010,2025-12-15,pricing,News,Data and voice prices increased 20-82%
40,EVT_0008,2025-12-18,infrastructure,NBE/EthSwitch,National real-time payment system


In [9]:
links = df[df['record_type'] == 'impact_link'].copy()
event_lookup = events.set_index('record_id')['original_text']
links['parent_event'] = links['parent_id'].map(event_lookup)
links[['record_id', 'parent_id', 'parent_event', 'related_indicator',
       'impact_direction', 'impact_magnitude', 'lag_months', 'evidence_basis', 'confidence']]


,record_id,parent_id,parent_event,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis,confidence
43,IMP_0001,EVT_0001,First major mobile money service in Ethiopia,ACC_OWNERSHIP,increase,high,12.0,literature,medium
44,IMP_0002,EVT_0001,First major mobile money service in Ethiopia,USG_TELEBIRR_USERS,increase,high,3.0,empirical,high
45,IMP_0003,EVT_0001,First major mobile money service in Ethiopia,USG_P2P_COUNT,increase,high,6.0,empirical,medium
46,IMP_0004,EVT_0002,End of state telecom monopoly,ACC_4G_COV,increase,medium,12.0,empirical,medium
47,IMP_0005,EVT_0002,End of state telecom monopoly,AFF_DATA_INCOME,decrease,medium,12.0,literature,medium
48,IMP_0006,EVT_0003,Second mobile money entrant,USG_MPESA_USERS,increase,high,3.0,empirical,high
49,IMP_0007,EVT_0003,Second mobile money entrant,ACC_MM_ACCOUNT,increase,medium,6.0,theoretical,medium
50,IMP_0008,EVT_0004,National biometric digital ID system,ACC_OWNERSHIP,increase,medium,24.0,literature,medium
51,IMP_0009,EVT_0004,National biometric digital ID system,GEN_GAP_ACC,decrease,medium,24.0,literature,medium
52,IMP_0010,EVT_0005,Birr float introduced,AFF_DATA_INCOME,increase,high,3.0,empirical,high


In [10]:
targets = df[df['record_type'] == 'target'].copy()
targets[['record_id', 'indicator', 'indicator_code', 'value_numeric', 'observation_date', 'source_name']]


,record_id,indicator,indicator_code,value_numeric,observation_date,source_name
30,REC_0031,Account Ownership Rate,ACC_OWNERSHIP,70.0,2025-12-31,NFIS-II Strategy
31,REC_0032,Fayda Digital ID Enrollment,ACC_FAYDA,90000000.0,2028-12-31,Fayda/NIDP
32,REC_0033,Female Mobile Money Account Share,GEN_MM_SHARE,50.0,2030-12-31,NBE


In [11]:
new_records = []